# PB03 — Read↔Img Similarity come Metrica di BCI Literacy

**Ipotesi**: i soggetti BCI-literate hanno pattern EEG simili tra `word_read` e `word_img` per la stessa parola.
Nei soggetti illiterate, l'immaginazione non attiva le stesse reti della lettura.

**Metrica**: correlazione di Pearson tra `word_read` e `word_img` (stessa parola, stesso soggetto),
mediata su tutte le parole → un singolo score di "fedeltà immaginativa" per soggetto.

**Validazione**: correla questo score con la `bacc` del soggetto nel task imagined speech.

In [ ]:
from pathlib import Path
import json

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists()),
    Path().resolve()
)

DATA_ROOT   = project_root / 'data' / 'raw_csv' / 'training_set'
CONFIGS     = project_root / 'configs' / 'label_schemes'
SFREQ = 256
N_CHAN = 61
N_SAMP = 384

with open(CONFIGS / 'label2idx.json') as f:
    word2idx = json.load(f)
WORDS = sorted(word2idx.keys())
print(f'Parole: {len(WORDS)} | DATA_ROOT: {DATA_ROOT}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
print('Import OK')

In [ ]:
# ============================================================
# CARICA BACC PER SOGGETTO — da PB00 o da W&B
# ============================================================
import pandas as pd

BACC_CSV = project_root / 'data' / 'interim' / 'subject_bacc_pipelineB.csv'

if BACC_CSV.exists():
    _df_bacc = pd.read_csv(BACC_CSV).dropna(subset=['bacc'])
    SUBJ_BACC = dict(zip(_df_bacc['subj_id'].astype(int), _df_bacc['bacc']))
    print(f'Caricati {len(SUBJ_BACC)} soggetti da {BACC_CSV.name}')
else:
    print('WARNING: subject_bacc_pipelineB.csv non trovato. Esegui PB00 prima.')
    SUBJ_BACC = {}

In [ ]:
# ============================================================
# CALCOLA READ↔IMG SIMILARITY PER SOGGETTO
# ============================================================

def read_img_similarity(subj_id: int, data_root: Path) -> float:
    """
    Per ogni parola: media delle epoche read e img su tutte le sessioni.
    Calcola correlazione di Pearson tra i due segnali medi (flatten).
    Ritorna la correlazione media su tutte le parole.
    """
    corrs = []
    sess_dirs = sorted(data_root.glob(f'P{subj_id:03d}_S*'))
    if not sess_dirs:
        return np.nan

    for word in WORDS:
        imgs, reads = [], []
        for sess_dir in sess_dirs:
            f_img  = sess_dir / f'{word}_img.csv'
            f_read = sess_dir / f'{word}_read.csv'
            if f_img.exists():
                x = pd.read_csv(f_img, header=None).values.astype(np.float32)
                if x.shape == (N_CHAN, N_SAMP):
                    imgs.append(x)
            if f_read.exists():
                x = pd.read_csv(f_read, header=None).values.astype(np.float32)
                if x.shape == (N_CHAN, N_SAMP):
                    reads.append(x)

        if not imgs or not reads:
            continue

        mean_img  = np.mean(imgs, axis=0).flatten()   # (61*384,)
        mean_read = np.mean(reads, axis=0).flatten()
        r, _ = stats.pearsonr(mean_img, mean_read)
        corrs.append(r)

    return float(np.mean(corrs)) if corrs else np.nan


# Test su un soggetto
sim = read_img_similarity(0, DATA_ROOT)
print(f'P000 read↔img similarity: {sim:.4f}')

In [ ]:
# ============================================================
# CALCOLA PER TUTTI I SOGGETTI
# ============================================================

all_subj = sorted(set(
    int(d.name.split('_')[0][1:]) for d in DATA_ROOT.iterdir() if d.is_dir()
))

similarities, subj_ids = [], []
for sid in tqdm(all_subj, desc='Soggetti'):
    sim = read_img_similarity(sid, DATA_ROOT)
    similarities.append(sim)
    subj_ids.append(sid)

df = pd.DataFrame({'subj': subj_ids, 'read_img_sim': similarities})
df['bacc'] = df['subj'].map(SUBJ_BACC)
print(df.describe())
print(f'\nSoggetti con similarity calcolata: {df["read_img_sim"].notna().sum()}')

In [ ]:
# ============================================================
# CORRELAZIONE SIMILARITY ↔ BACC
# ============================================================

df_valid = df.dropna(subset=['read_img_sim', 'bacc'])

if len(df_valid) > 5:
    rho, p = stats.spearmanr(df_valid['read_img_sim'], df_valid['bacc'])
    print(f'Spearman ρ = {rho:.3f}  p = {p:.4f}  (n={len(df_valid)})')

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(df_valid['read_img_sim'], df_valid['bacc'], alpha=0.7, edgecolors='k', linewidths=0.5)
    ax.set_xlabel('Read↔Img similarity (Pearson r)')
    ax.set_ylabel('bacc imagined speech')
    ax.set_title(f'Fedeltà immaginativa vs BCI performance  ρ={rho:.3f}  p={p:.4f}')
    plt.tight_layout()
    plt.savefig(project_root / 'figures' / 'PB03_readimg_similarity_vs_bacc.png', dpi=150)
    plt.show()
else:
    print('TODO: caricare SUBJ_BACC per la correlazione')

In [ ]:
# ============================================================
# DISTRIBUZIONE SIMILARITY PER SOGGETTO
# ============================================================

df_sorted = df.sort_values('read_img_sim', ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['steelblue' if not np.isnan(b) and b > 0.30 else 'tomato'
          for b in df_sorted['bacc']]
ax.bar(range(len(df_sorted)), df_sorted['read_img_sim'], color=colors, alpha=0.8)
ax.set_xlabel('Soggetto (ordinato per similarity)')
ax.set_ylabel('Read↔Img similarity')
ax.set_title('Fedeltà immaginativa per soggetto (blu=literate, rosso=illiterate)')
plt.tight_layout()
plt.savefig(project_root / 'figures' / 'PB03_similarity_per_subject.png', dpi=150)
plt.show()